In [ ]:
import os, gzip, re, math
from collections import defaultdict, deque, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

PATH_DRUGBANK_CLEAN = "../../data/01_clean/drugbank_pralsetinib_proteins_cleaned.csv"
PATH_GOA_GAF_GZ     = "../../data/00_raw/goa_human_annotations.gaf.gz"
PATH_GO_OBO         = "../../data/00_raw/go-basic.obo"

OUT_FULL = "figures/rah/pralsetinib_full_go_bp_enrichment.csv"
OUT_TOP  = "figures/rah/pralsetinib_top_go_bp_terms.csv"
OUT_NET  = "figures/rah/pralsetinib_go_enrichment_map.png"
OUT_BAR  = "figures/rah/pralsetinib_go_top_terms_clustered.png"

for p in [PATH_DRUGBANK_CLEAN, PATH_GOA_GAF_GZ, PATH_GO_OBO]:
    assert os.path.exists(p), f"Missing file: {p}"
print("Inputs found ✅")

AssertionError: Missing file: data/01_clean/drugbank_pralsetinib_proteins_cleaned.csv

In [4]:
def parse_go_obo_full(obo_path: str):
    parents = defaultdict(set)
    children = defaultdict(set)
    name = {}
    namespace = {}

    term_id = None
    term_name = None
    term_ns = None
    is_obsolete = False

    def commit():
        nonlocal term_id, term_name, term_ns, is_obsolete
        if term_id and (not is_obsolete):
            if term_name is not None:
                name[term_id] = term_name
            if term_ns is not None:
                namespace[term_id] = term_ns

    with open(obo_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip() == "[Term]":
                commit()
                term_id = None
                term_name = None
                term_ns = None
                is_obsolete = False
                continue
            if line.startswith("id: GO:"):
                term_id = line.split("id:")[1].strip()
                continue
            if line.startswith("name:"):
                term_name = line.split("name:")[1].strip()
                continue
            if line.startswith("namespace:"):
                term_ns = line.split("namespace:")[1].strip()
                continue
            if line.startswith("is_obsolete:"):
                is_obsolete = (line.split("is_obsolete:")[1].strip().lower() == "true")
                continue
            if term_id and (not is_obsolete) and line.startswith("is_a: GO:"):
                parent = line.split("is_a:")[1].split("!")[0].strip()
                parents[term_id].add(parent)
                children[parent].add(term_id)

    commit()
    return parents, children, name, namespace

parents, children, go_name, go_ns = parse_go_obo_full(PATH_GO_OBO)
bp_terms = {go for go, ns in go_ns.items() if ns == "biological_process"}

print("BP terms:", len(bp_terms))

FileNotFoundError: [Errno 2] No such file or directory: 'data/00_raw/go-basic.obo'